# ART SLIDE — 美術館作品を収集してGitHubに直接反映

シカゴ美術館とメトロポリタン美術館の公式APIから、パブリックドメインと確認できた作品の画像・作品情報を取得します。**GitHubへの保存まで全自動**です。

## 初回セットアップ
1. GitHubの Fine-grained personal access token を用意します。**Repository access: Only select repositories → `ryouy/slides`**、**Repository permissions → Contents → Read and write** を設定してください。
2. Colabの左側にある鍵アイコン（シークレット）を開き、名前を **`GITHUB_TOKEN`** にして登録します。
3. **ノートブックからのアクセスを許可**をオンにしてください。トークンはノートブックに記入せず、実行結果にも表示されません。
4. 下のセルを順番に実行します。

## 保存するもの
- `public/artworks/`：ゲーム画像とサムネイル（WebP）
- `src/games/art-slide/data/additional-artworks.ts`：これまでの追加分を含む作品一覧
- `src/games/art-slide/data/artworks-research.json`：出典・説明文ライセンス・関連情報

画像と作品一覧は**1つのGitコミット**で更新します。アプリは追加作品ファイルをすでに読み込むよう設定済みです。元の `artworks.ts` とデイリー用の固定リストは変更しません。毎回、GitHub上の既存作品と照合し、重複を防ぎます。

**ライセンス**：画像は公式APIのパブリックドメイン判定を条件に収集します。シカゴ美術館の解説文は画像とは別に**CC BY 4.0**として帰属情報を保存します。自動収集された文章の内容と権利表示は、公開後も適宜確認してください。

In [ ]:
%pip -q install requests pillow beautifulsoup4


In [ ]:
# ART SLIDE Colab collector — official museum APIs; single atomic GitHub commit.
# Configuration (you may adjust the targets before running).
TARGET_AIC = 100
TARGET_MET = 40
AIC_PAGES = 12
MET_CANDIDATES_PER_SEARCH = 130
MET_SEARCH_TERMS = ["Monet", "Van Gogh", "Renoir", "Vermeer", "Hokusai", "Degas", "Sargent", "Turner", "Rembrandt", "Cézanne"]
MIN_YEAR = 1300
MAX_YEAR = 1925
GAME_EDGE = 1200
THUMB_EDGE = 420
MIN_IMAGE_EDGE = 450
DOWNLOAD_ZIP_BACKUP = True
REPO = "ryouy/slides"
BRANCH = "main"

import base64, html, io, json, random, re, time, unicodedata, zipfile
from datetime import date
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps
from google.colab import userdata, files

API = "https://api.github.com/repos/" + REPO
AIC_API = "https://api.artic.edu/api/v1"
MET_API = "https://collectionapi.metmuseum.org/public/collection"
CC0 = "https://creativecommons.org/publicdomain/zero/1.0/"
CC_BY = "https://creativecommons.org/licenses/by/4.0/"
session = requests.Session()
session.headers.update({"User-Agent": "ART-SLIDE-Colab/1.0 (sequential requests)",
                        "AIC-User-Agent": "ART-SLIDE-Colab/1.0"})

def json_api(url, params=None, pause=.65):
    for attempt in range(5):
        time.sleep(pause)
        try:
            r = session.get(url, params=params, timeout=(15, 60))
            if r.status_code in (429, 500, 502, 503, 504):
                wait = min(50, 3 * 2 ** attempt)
                print("Museum API:", r.status_code, "retrying in", wait, "seconds")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == 4: raise
            time.sleep(min(40, 3 * 2 ** attempt))
    raise RuntimeError("Museum API unavailable: " + url)

def plain_text(value):
    soup = BeautifulSoup(html.unescape(str(value or "")), "html.parser")
    for tag in soup(["script", "style"]): tag.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ", strip=True)).strip()

def normalized(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    return re.sub(r"[^a-z0-9]+", " ", "".join(c for c in value if not unicodedata.combining(c)).lower()).strip()

def year_int(value):
    m = re.search(r"(?<!\d)(1[3-9]\d\d|20[0-2]\d)(?!\d)", str(value or ""))
    return int(m.group(1)) if m else None

def era(y):
    return "Before 1700" if y < 1700 else "1700s" if y < 1800 else "1800–1849" if y < 1850 else "1850–1879" if y < 1880 else "1880–1899" if y < 1900 else "1900s"

def theme(title, medium="", subjects="", japan=False):
    if japan: return "Japanese Prints"
    s = normalized(title + " " + medium + " " + subjects)
    for label, pattern in [
        ("Myth & Religion", r"saint|christ|madonna|angel|religio|myth|holy"),
        ("Still Life & Flowers", r"still life|flower|rose|vase|fruit|bouquet"),
        ("Portrait", r"portrait|self portrait|madame|lady|girl|woman|man"),
        ("Sea & Water", r"sea|river|water|ocean|beach|coast|wave|harbor|port|boat"),
        ("City & Street", r"city|town|street|avenue|bridge|piazza|paris"),
        ("Landscape", r"landscape|field|forest|mountain|tree|garden|hill|valley"),
    ]:
        if re.search(pattern, s): return label
    return "People & Daily Life"

def title_key(artist, title):
    return (normalized(artist), normalized(title))

def summary(title, artist, year, medium, other="", tags=None):
    bits = [title + " is a work by " + artist + "."]
    if year: bits.append("The museum dates this work to " + year + ".")
    if medium: bits.append("Its medium is " + medium + ".")
    if other: bits.append("The catalogue associates it with " + other + ".")
    if tags: bits.append("Its recorded subjects include " + ", ".join(tags[:5]) + ".")
    return " ".join(bits)

def download_webps(url):
    host = urlparse(url).hostname
    if host not in ("www.artic.edu", "images.metmuseum.org"):
        raise ValueError("Unapproved image host")
    time.sleep(.4)
    with session.get(url, stream=True, timeout=(15, 90)) as resp:
        resp.raise_for_status()
        parts, total = [], 0
        for chunk in resp.iter_content(65536):
            total += len(chunk)
            if total > 30 * 1024 * 1024: raise ValueError("Image too large")
            parts.append(chunk)
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(b"".join(parts)))).convert("RGB")
    if min(im.size) < MIN_IMAGE_EDGE or max(im.size) / min(im.size) > 3.8:
        raise ValueError("Image unsuitable for slider")
    def webp(edge, quality):
        x = im.copy()
        x.thumbnail((edge, edge), Image.Resampling.LANCZOS)
        out = io.BytesIO()
        x.save(out, "WEBP", quality=quality, method=5)
        return out.getvalue(), x.size
    full, dimensions = webp(GAME_EDGE, 82)
    thumb, _ = webp(THUMB_EDGE, 76)
    return full, thumb, dimensions

def record(work_id, title, artist, year, start, end, medium, museum, source, image_url, dims, kind, description, desc_source=None):
    return {
        "id": work_id, "title": title, "artist": artist,
        "artistSort": normalized(artist).split(" ")[-1] + " " + normalized(artist),
        "year": year, "yearStart": start, "yearEnd": end,
        "era": era(start), "theme": kind, "medium": medium,
        "imageUrl": "/artworks/" + work_id + ".webp",
        "thumbnailUrl": "/artworks/" + work_id + "-thumb.webp",
        "width": dims[0], "height": dims[1], "museum": museum,
        "sourceUrl": source, "license": "CC0 1.0 (public-domain image)",
        "licenseUrl": CC0,
        "licenseBasis": museum + ": public-domain status verified via official API",
        "sourceImageUrl": image_url, "verifiedAt": date.today().isoformat(),
        "description": description,
        "descriptionKind": "museum_text" if desc_source else "metadata_summary",
        **({
          "descriptionAttribution": "Art Institute of Chicago",
          "descriptionSourceUrl": desc_source, "descriptionLicense": "CC BY 4.0",
          "descriptionLicenseUrl": CC_BY,
          "descriptionModifications": "Original museum HTML converted to plain text; whitespace normalized."
        } if desc_source else {}),
    }

token = (userdata.get("GITHUB_TOKEN") or "").strip()
if not token:
    raise RuntimeError("Colab Secrets (key icon): create GITHUB_TOKEN and enable notebook access.")
github = requests.Session()
github.headers.update({
    "Authorization": "Bearer " + token,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "ART-SLIDE-Colab/1.0",
})

def gh(method, path, payload=None):
    for attempt in range(5):
        r = github.request(method, API + path, json=payload, timeout=(15, 150))
        if r.status_code in (429, 500, 502, 503, 504) or (r.status_code == 403 and r.headers.get("Retry-After")):
            if attempt == 4: raise RuntimeError("GitHub temporarily unavailable")
            time.sleep(int(r.headers.get("Retry-After") or min(60, 3 * 2 ** attempt)))
            continue
        if not r.ok:
            try: detail = r.json().get("message", "")
            except ValueError: detail = ""
            raise RuntimeError("GitHub " + method + " " + path + ": HTTP " + str(r.status_code) + " " + detail[:180])
        return r.json()
    raise RuntimeError("GitHub API error")

def repo_text(path, optional=False):
    r = github.get(API + "/contents/" + path, params={"ref": BRANCH}, timeout=(15, 50))
    if optional and r.status_code == 404: return ""
    r.raise_for_status()
    data = r.json()
    return base64.b64decode(data["content"]).decode("utf-8")

def parse_list(path, export_name, optional=False):
    text = repo_text(path, optional)
    if not text and optional: return []
    match = re.search(r"export const " + export_name + r": Artwork\[\]\s*=\s*(\[.*?\]);", text, re.S)
    if not match: raise ValueError("Cannot parse " + path + ": refusing unsafe duplicate import")
    return json.loads(match.group(1))

# Read both original and previous additions so repeated runs are safe.
base = parse_list("src/games/art-slide/data/artworks.ts", "ARTWORKS")
old_extra = parse_list("src/games/art-slide/data/additional-artworks.ts", "NEW_ARTWORKS", True)
old_research_text = repo_text("src/games/art-slide/data/artworks-research.json", True)
old_research = json.loads(old_research_text) if old_research_text else []
seen_titles = {title_key(x["artist"], x["title"]) for x in base + old_extra}
seen_sources = {x["sourceUrl"] for x in base + old_extra}
print("Existing:", len(base), "base works;", len(old_extra), "previous additions.")

works, research, images = [], [], {}
def add(work, full, thumb, meta):
    works.append(work)
    seen_titles.add(title_key(work["artist"], work["title"]))
    seen_sources.add(work["sourceUrl"])
    images["public/artworks/" + work["id"] + ".webp"] = full
    images["public/artworks/" + work["id"] + "-thumb.webp"] = thumb
    research.append(meta)
    print("Added", len(works), work["museum"], "—", work["title"][:65])

aic_count = 0
fields = ",".join(["id","title","artist_title","date_display","date_start","date_end",
                   "image_id","is_public_domain","artwork_type_title","classification_title",
                   "medium_display","style_title","place_of_origin","subject_titles",
                   "description","short_description"])
for page in range(1, AIC_PAGES + 1):
    if aic_count >= TARGET_AIC: break
    result = json_api(AIC_API + "/artworks/search", {
        "query[term][is_public_domain]": "true",
        "fields": fields, "page": page, "limit": 100,
    })
    items = result.get("data") or []
    if not items: break
    for a in items:
        if aic_count >= TARGET_AIC: break
        ident = a.get("id")
        if not ident or a.get("is_public_domain") is not True or not a.get("image_id"): continue
        title, artist = str(a.get("title") or "").strip(), str(a.get("artist_title") or "").strip()
        if not title or not artist: continue
        url = "https://www.artic.edu/artworks/" + str(ident)
        if url in seen_sources or title_key(artist, title) in seen_titles: continue
        medium = str(a.get("medium_display") or "")
        classification = normalized(str(a.get("artwork_type_title") or a.get("classification_title") or "") + " " + medium)
        if not re.search(r"painting|print|oil|watercolor|tempera|woodblock|etching", classification): continue
        start = year_int(a.get("date_start")) or year_int(a.get("date_display"))
        if not start or start < MIN_YEAR or start > MAX_YEAR: continue
        end = year_int(a.get("date_end")) or start
        year = str(a.get("date_display") or start)
        curator = plain_text(a.get("description") or a.get("short_description"))
        subjects = [str(v) for v in (a.get("subject_titles") or []) if v][:8]
        source_image = "https://www.artic.edu/iiif/2/" + str(a["image_id"]) + "/full/1200,/0/default.jpg"
        work_id = "aic-" + str(ident)
        try: full, thumb, dims = download_webps(source_image)
        except (OSError, ValueError, requests.RequestException) as exc:
            print("Skipping image", work_id, type(exc).__name__)
            continue
        description = curator or summary(title, artist, year, medium, str(a.get("style_title") or ""), subjects)
        work = record(work_id, title, artist, year, start, end, medium,
                      "Art Institute of Chicago", url, source_image, dims,
                      theme(title, medium, " ".join(subjects)), description, url if curator else None)
        add(work, full, thumb, {
            "id": work_id, "museumId": str(ident), "sourceUrl": url,
            "descriptionKind": work["descriptionKind"],
            "descriptionLicense": "CC BY 4.0" if curator else "metadata-derived original summary",
            "descriptionLicenseUrl": CC_BY if curator else None,
            "descriptionAttribution": "Art Institute of Chicago" if curator else None,
            "descriptionModifications": work.get("descriptionModifications"),
            "style": a.get("style_title"), "place": a.get("place_of_origin"), "subjects": subjects,
        })
        aic_count += 1
    print("AIC page", page, "completed:", aic_count, "accepted")

met_count = 0
for term in MET_SEARCH_TERMS:
    if met_count >= TARGET_MET: break
    try: results = json_api(MET_API + "/v1/search", {"q": term, "hasImages": "true"}, pause=.6)
    except requests.RequestException as exc:
        print("Skipping Met search", term, type(exc).__name__)
        continue
    for ident in (results.get("objectIDs") or [])[:MET_CANDIDATES_PER_SEARCH]:
        if met_count >= TARGET_MET: break
        url = "https://www.metmuseum.org/art/collection/search/" + str(ident)
        if url in seen_sources: continue
        try: a = json_api(MET_API + "/v1/objects/" + str(ident), pause=.4)
        except requests.RequestException: continue
        if a.get("isPublicDomain") is not True: continue
        title, artist = str(a.get("title") or "").strip(), str(a.get("artistDisplayName") or "").strip()
        source_image = str(a.get("primaryImageSmall") or a.get("primaryImage") or "")
        if not title or not artist or not source_image or title_key(artist, title) in seen_titles: continue
        medium = str(a.get("medium") or "")
        classification = normalized(str(a.get("classification") or "") + " " + str(a.get("objectName") or "") + " " + medium)
        if not re.search(r"painting|print|oil|woodblock|watercolor|tempera|etching", classification): continue
        start = year_int(a.get("objectBeginDate")) or year_int(a.get("objectDate"))
        if not start or start < MIN_YEAR or start > MAX_YEAR: continue
        end = year_int(a.get("objectEndDate")) or start
        year = str(a.get("objectDate") or start)
        tags = [str(v["term"]) for v in (a.get("tags") or []) if isinstance(v,dict) and v.get("term")][:8]
        culture = str(a.get("culture") or a.get("period") or "")
        japan = "japan" in normalized(culture) and ("woodblock" in normalized(medium) or "print" in classification)
        work_id = "met-" + str(ident)
        try: full, thumb, dims = download_webps(source_image)
        except (OSError, ValueError, requests.RequestException):
            print("Skipping image", work_id)
            continue
        description = summary(title, artist, year, medium, culture, tags)
        work = record(work_id, title, artist, year, start, end, medium,
                      "The Metropolitan Museum of Art", url, source_image, dims,
                      theme(title, medium, " ".join(tags), japan=japan), description)
        add(work, full, thumb, {"id": work_id, "museumId": str(ident), "sourceUrl": url,
            "descriptionKind": "metadata_summary", "descriptionLicense": "original metadata-derived summary",
            "culture": culture, "tags": tags, "department": a.get("department")})
        met_count += 1
    print("Met term", term, "completed:", met_count, "accepted")

if not works:
    print("No new artworks. Increase the search limits or use other artist terms.")
else:
    merged = old_extra + works
    merged_research = {x["id"]: x for x in old_research if isinstance(x,dict) and x.get("id")}
    merged_research.update({x["id"]: x for x in research})
    manifest = ("import type { Artwork } from '../types';\n\n"
                "// Auto-generated by notebooks/ART_SLIDE_artwork_collector.ipynb.\n"
                "export const NEW_ARTWORKS: Artwork[] = "
                + json.dumps(merged, ensure_ascii=False, indent=2) + ";\n")
    research_text = json.dumps(list(merged_research.values()), ensure_ascii=False, indent=2) + "\n"
    # A single fast-forward commit updates both images and manifests together.
    meta = gh("GET", "")
    if meta.get("full_name") != REPO: raise RuntimeError("Wrong GitHub repository")
    perms = meta.get("permissions") or {}
    if perms and not (perms.get("push") or perms.get("admin")):
        raise RuntimeError("GITHUB_TOKEN must have Contents: Read and write permission.")
    ref = gh("GET", "/git/ref/heads/" + BRANCH)
    parent = ref["object"]["sha"]
    parent_tree = gh("GET", "/git/commits/" + parent)["tree"]["sha"]
    tree = []
    for idx, (path, blob) in enumerate(sorted(images.items()), 1):
        sha = gh("POST", "/git/blobs", {"content": base64.b64encode(blob).decode("ascii"), "encoding": "base64"})["sha"]
        tree.append({"path": path, "mode": "100644", "type": "blob", "sha": sha})
        if idx % 20 == 0 or idx == len(images): print("GitHub blobs:", idx, "/", len(images))
        time.sleep(.16)
    for path, value in {
        "src/games/art-slide/data/additional-artworks.ts": manifest,
        "src/games/art-slide/data/artworks-research.json": research_text,
    }.items():
        tree.append({"path": path, "mode": "100644", "type": "blob", "content": value})
    new_tree = gh("POST", "/git/trees", {"base_tree": parent_tree, "tree": tree})["sha"]
    commit = gh("POST", "/git/commits", {
        "message": "Add " + str(len(works)) + " museum artworks from Colab",
        "tree": new_tree, "parents": [parent],
    })["sha"]
    gh("PATCH", "/git/refs/heads/" + BRANCH, {"sha": commit, "force": False})
    print("Successfully committed", len(works), "new artworks.")
    print("https://github.com/" + REPO + "/commit/" + commit)
    print("Existing base artworks and the frozen daily rotation were not modified.")
    if DOWNLOAD_ZIP_BACKUP:
        with zipfile.ZipFile("art_slide_expansion.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=5) as z:
            for path, blob in images.items(): z.writestr(path, blob)
            z.writestr("src/games/art-slide/data/additional-artworks.ts", manifest)
            z.writestr("src/games/art-slide/data/artworks-research.json", research_text)
            z.writestr("artworks_new.json", json.dumps(works, ensure_ascii=False, indent=2))
        files.download("art_slide_expansion.zip")


## 実行後

最後にGitHubのコミットURLが表示されます。GitHubの `main` に直接書き込める権限があれば、連携デプロイが起動し、新しい作品がギャラリー・ランダムプレイで使えるようになります。

`TARGET_AIC` と `TARGET_MET` の数値を小さくして試すこともできます。GitHubのブランチ保護により直接更新できない場合は、書き込みが拒否されます（無理な上書きはしません）。

作品テキストは、シカゴ美術館の公開解説がある場合はそのテキストを利用し、それ以外は公式メタデータに基づく英語の説明を作成します。7言語への詳細解説の自動翻訳は含まれません。